# MOUSE CORE · BGE-M3 + strukturální pilot v2 · devět autorských pohádek + lidová reference

Základní soubor zůstává devíti původními kartami: **Myš, Ryba, Kocour, Lemur, Slon, Páv, Čáp, Kozel a Zajíc**.

Nově je přidána samostatná referenční lidová pohádka **O kohoutkovi a slepičce**. Není započítána jako desátá autorská karta. Její vektor a strukturální analýza se ukládají odděleně a počítá se pouze její podobnost vůči každé z devíti autorských pohádek.

Metodické hranice:
- Každý text zůstává jedním celkem, bez vlastního dělení na významové segmenty.
- U devíti autorských textů se odstraní pouze technické řádky začínající Karta: nebo Text:.
- Lidová pohádka se použije v podobě dodané uživatelkou, bez dalších úprav.
- BGE-M3 zpracovává pouze text. Ilustrace nejsou obrazově analyzovány.
- Podobnost sama o sobě nedokazuje významovou funkci ani porozumění.

V Google Colab zvol GPU, pokud je dostupné, a spusť **Běh → Spustit vše**. Model se stáhne do dočasného výpočetního prostředí, nikoli do telefonu.

In [ ]:
!pip -q install -U FlagEmbedding pandas matplotlib

In [ ]:
import json, re, math, unicodedata
from pathlib import Path
from urllib.request import urlopen
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

REPO = "https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main"
MODEL_NAME = "BAAI/bge-m3"
ANGLE_LENSES = ["Analytický", "Kulturní", "Filozofický", "Naivní", "Jazykový", "Metaforický", "Asociační", "Introspektivní", "Ironický", "Groteskní", "Absurdní", "Surrealistický"]
CARDS = {
    "mys": {"name": "Myš", "text": "doma/primarni_data/kriva_pohadka/mys.txt"},
    "ryba": {"name": "Ryba", "text": "doma/primarni_data/kriva_pohadka/ryba.txt"},
    "kocour": {"name": "Kocour", "text": "doma/primarni_data/kriva_pohadka/kocour.txt"},
    "lemur": {"name": "Lemur", "text": "doma/primarni_data/kriva_pohadka/lemur.txt"},
    "slon": {"name": "Slon", "text": "doma/primarni_data/kriva_pohadka/slon.txt"},
    "pav": {"name": "Páv", "text": "doma/primarni_data/kriva_pohadka/pav.txt"},
    "cap": {"name": "Čáp", "text": "doma/primarni_data/kriva_pohadka/cap.txt"},
    "kozel": {"name": "Kozel", "text": "doma/primarni_data/kriva_pohadka/kozel.txt"},
    "zajic": {"name": "Zajíc", "text": "doma/primarni_data/kriva_pohadka/zajic.txt"}
}
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("Zařízení:", "GPU" if torch.cuda.is_available() else "CPU")
print("Výstup:", OUT)

## 1 · Načtení původních dat

Stahujeme přesné zdrojové soubory z repozitáře. Nepřepisujeme je, nepřekládáme a nekrátíme.

In [ ]:
documents = []
input_normalization = {}
for card_id, info in CARDS.items():
    raw = urlopen(f"{REPO}/{info['text']}").read().decode("utf-8-sig")
    # Odstraníme pouze celé technické řádky Karta:/Text:, kdekoli v souboru.
    # Všechen ostatní literární text zůstává zachovaný.
    removed_lines = re.findall(r"(?m)^[ \t]*(?:Karta|Text):[^\r\n]*(?:\r?\n|$)", raw)
    model_text = re.sub(r"(?m)^[ \t]*(?:Karta|Text):[^\r\n]*(?:\r?\n|$)", "", raw).strip()
    assert model_text, f"Po očištění zůstal prázdný text: {card_id}"
    input_normalization[card_id] = {
        "removed_metadata_line_count": len(removed_lines),
        "removed_metadata_labels": [line.split(":", 1)[0].strip() for line in removed_lines],
        "rule": "Odstraněny pouze celé technické řádky Karta: a Text:; ostatní text zachován",
        "story_text_preserved_as_one_input": True
    }
    documents.append({
        "card_id": card_id,
        "card_name": info["name"],
        "source_file": info["text"],
        "original_text": model_text
    })
    print(f"{info['name']}: {len(model_text)} znaků | odstraněno technických řádků: {len(removed_lines)}")
assert len(documents) == 9
assert len({d["card_id"] for d in documents}) == 9
print("Pořadí karet:", [d["card_name"] for d in documents])
print("Normalizace:", input_normalization)

## 2 · Strukturální měření bez segmentace

Tato část neříká, co slovo znamená. Měří, co lze ověřit přímo v textu: kolikrát se jazykový prvek vrací, jaká je pestrost slovníku, jaké znaky se opakují a v jakém bezprostředním okolí se výskyty nacházejí.

Kontexty jsou pouze okolní text kolem konkrétního výskytu. Nejsou to automaticky významové jednotky.

In [ ]:
WORD_RE = re.compile(r"[^\W\d_]+(?:[’'-][^\W\d_]+)*", re.UNICODE)
BIGRAM_RE = None

def contexts_for(text, spans, radius=75):
    out = []
    for start, end, token in spans:
        out.append({
            "token": token,
            "start_char": start,
            "end_char": end,
            "context": text[max(0, start-radius):min(len(text), end+radius)]
        })
    return out

def analyze_structure(doc):
    text = doc["original_text"]
    matches = list(WORD_RE.finditer(text))
    words = [m.group(0).casefold() for m in matches]
    counts = Counter(words)
    bigrams = Counter(zip(words, words[1:]))
    punct = Counter(ch for ch in text if unicodedata.category(ch).startswith("P"))
    total = len(words)
    unique = len(counts)
    probs = [v/total for v in counts.values()] if total else []
    entropy = -sum(p*math.log2(p) for p in probs) if probs else 0.0
    max_entropy = math.log2(unique) if unique > 1 else 0.0
    repeated = {w:c for w,c in counts.items() if c > 1}
    repeated_spans = [(m.start(), m.end(), m.group(0)) for m in matches if counts[m.group(0).casefold()] > 1]
    repeated_bigrams = [{"bigram": list(k), "count": v} for k,v in bigrams.items() if v > 1]
    punctuation_contexts = []
    for i,ch in enumerate(text):
        if ch in ".!?…,:;":
            punctuation_contexts.append({
                "mark": ch, "start_char": i,
                "context": text[max(0,i-75):min(len(text),i+76)]
            })
    sentence_like = len(re.findall(r"[.!?…]+", text))
    paragraphs = [p for p in re.split(r"\n\s*\n", text) if p.strip()]
    return {
        "card_id": doc["card_id"],
        "card_name": doc["card_name"],
        "source_file": doc["source_file"],
        "characters_including_spaces": len(text),
        "characters_without_whitespace": sum(not c.isspace() for c in text),
        "word_tokens": total,
        "unique_word_forms_casefolded": unique,
        "type_token_ratio": round(unique/total, 6) if total else 0,
        "hapax_count": sum(1 for c in counts.values() if c == 1),
        "hapax_ratio": round(sum(1 for c in counts.values() if c == 1)/unique, 6) if unique else 0,
        "repeated_token_types": len(repeated),
        "token_share_from_repeated_types": round(sum(c for c in counts.values() if c > 1)/total, 6) if total else 0,
        "word_frequency": counts.most_common(30),
        "repeated_word_occurrences_with_context": contexts_for(text, repeated_spans),
        "repeated_adjacent_bigrams": repeated_bigrams,
        "punctuation_counts": dict(punct),
        "selected_punctuation_contexts": punctuation_contexts,
        "sentence_ending_mark_count": sentence_like,
        "paragraph_count": len(paragraphs),
        "word_frequency_entropy_bits": round(entropy, 6),
        "normalized_word_frequency_entropy": round(entropy/max_entropy, 6) if max_entropy else 0
    }

structure = [analyze_structure(d) for d in documents]
summary_rows = [{
    "card": x["card_name"],
    "characters": x["characters_including_spaces"],
    "word_tokens": x["word_tokens"],
    "unique_forms": x["unique_word_forms_casefolded"],
    "type_token_ratio": x["type_token_ratio"],
    "repeated_types": x["repeated_token_types"],
    "repeated_token_share": x["token_share_from_repeated_types"],
    "paragraphs": x["paragraph_count"],
    "sentence_end_marks": x["sentence_ending_mark_count"]
} for x in structure]
display(pd.DataFrame(summary_rows).set_index("card"))


## 3 · Skutečné embeddingy celých pohádek

Každý text je předán modelu jako jedna položka. Model může interně tokenizovat text, ale my z toho neděláme vlastní sémantické segmenty.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(MODEL_NAME, use_fp16=torch.cuda.is_available())
whole_texts = [d["original_text"] for d in documents]
encoded = model.encode(
    whole_texts,
    batch_size=3,
    max_length=8192,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)
vectors = np.asarray(encoded["dense_vecs"], dtype=np.float32)
print("Počet vektorů:", len(vectors))
print("Dimenze:", vectors.shape[1])
assert vectors.shape[0] == len(documents) == 9

## 4 · Geometrie devíti reprezentací

Počítáme párové vztahy v prostoru embeddingů. Kosinová podobnost měří orientační podobnost směrů vektorů. Úhlová vzdálenost převádí kosinus na úhel. Eukleidovská vzdálenost měří přímou vzdálenost vektorů.

Žádná z těchto hodnot sama o sobě není měřítkem lidského porozumění. Výsledky platí pro tento model, jeho verzi a tento způsob reprezentace.

In [ ]:
def cosine(a, b):
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den else float("nan")

n = len(documents)
geometry = []
cos_matrix = np.zeros((n,n), dtype=float)
euclidean_matrix = np.zeros((n,n), dtype=float)
angular_matrix = np.zeros((n,n), dtype=float)

for i in range(n):
    for j in range(n):
        c = cosine(vectors[i], vectors[j])
        cos_matrix[i,j] = c
        euclidean_matrix[i,j] = float(np.linalg.norm(vectors[i]-vectors[j]))
        angular_matrix[i,j] = float(np.arccos(np.clip(c,-1,1)))

for i in range(n):
    for j in range(i+1,n):
        geometry.append({
            "card_a": documents[i]["card_id"],
            "card_b": documents[j]["card_id"],
            "cosine_similarity": float(cos_matrix[i,j]),
            "cosine_distance": float(1-cos_matrix[i,j]),
            "angular_distance_radians": float(angular_matrix[i,j]),
            "angular_distance_degrees": float(np.degrees(angular_matrix[i,j])),
            "euclidean_distance": float(euclidean_matrix[i,j])
        })

labels = [d["card_name"] for d in documents]
display(pd.DataFrame(cos_matrix, index=labels, columns=labels).round(5))
display(pd.DataFrame(geometry).sort_values("cosine_similarity", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))
im = ax.imshow(cos_matrix, vmin=-1, vmax=1)
ax.set_xticks(range(n), labels, rotation=45, ha="right")
ax.set_yticks(range(n), labels)
ax.set_title("BGE-M3 · kosinová podobnost devíti celých pohádek")
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{cos_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

# Heatmapa zobrazuje párovou podobnost všech devíti textů.
# Nejde o mapu významu ani o důkaz porozumění modelu.

## 5 · Export do MOUSE CORE

Vzniknou tři soubory:
- mouse_core_bge_m3_vectors_v2.json – devět skutečných embeddingů pro Vector Lab.
- mouse_core_bge_m3_analysis_v2.json – strukturální měření, kontexty opakování, párová geometrie a metadata.
- mouse_core_bge_m3_manifest_v2.json – popis běhu, modelu a zdrojů.

Původní texty jsou přiložené jako zdrojové podklady po odstranění pouze technických řádků Karta: a Text:.

In [ ]:
vector_rows = []
for doc, vec in zip(documents, vectors):
    vector_rows.append({
        "id": doc["card_id"] + "-full",
        "card_id": doc["card_id"],
        "card_name": doc["card_name"],
        "kind": "full_story",
        "text": doc["original_text"],
        "source_file": doc["source_file"],
        "illustration_file": None,
        "illustration_url": None,
        "model": MODEL_NAME,
        "vector": [float(v) for v in vec]
    })

manifest = {
    "experiment": "MOUSE CORE BGE-M3 + structural pilot",
    "model": MODEL_NAME,
    "model_type": "text_embedding",
    "language": "Czech",
    "cards": [d["card_id"] for d in documents],
    "angle_lenses_defined_by_method": ANGLE_LENSES,
    "angle_lenses_scored_automatically": False,
    "item_count": len(vector_rows),
    "text_cleaning": "Only full lines beginning Karta: or Text: were removed; literary content retained.",
    "vector_dimension": int(vectors.shape[1]),
    "source_repository": "https://github.com/artmisanovak-prog/generative_gen",
    "segmentation": "none; each complete original story is one embedding input",
    "input_normalization": input_normalization,
    "geometry_scope": "pairwise geometry of nine complete-story vectors in this pilot sample; not a general semantic topology",
    "illustrations": "not included in this text-only embedding run; no image embeddings",
    "limitations": [
        "Similarity is not proof of semantic understanding.",
        "Lexical counts do not determine the function or meaning of a repeated element.",
        "Nine stories are an expanded pilot, not a statistically representative sample."
    ]
}

analysis_payload = {
    "schema": "mouse-core-bge-m3-analysis-v1",
    "manifest": manifest,
    "structural_analysis": structure,
    "embedding_geometry": {
        "model": MODEL_NAME,
        "labels": labels,
        "cosine_similarity_matrix": cos_matrix.tolist(),
        "euclidean_distance_matrix": euclidean_matrix.tolist(),
        "angular_distance_matrix_radians": angular_matrix.tolist(),
        "pairwise": geometry,
        "interpretation_note": "Only pairwise relations among these nine complete-story vectors are measured. No general topology is inferred."
    }
}

vectors_path = OUT / "mouse_core_bge_m3_vectors_v2.json"
analysis_path = OUT / "mouse_core_bge_m3_analysis_v2.json"
manifest_path = OUT / "mouse_core_bge_m3_manifest_v2.json"

vectors_path.write_text(json.dumps(vector_rows, ensure_ascii=False), encoding="utf-8")
analysis_path.write_text(json.dumps(analysis_payload, ensure_ascii=False, indent=2), encoding="utf-8")
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("Uloženo:")
print(vectors_path)
print(analysis_path)
print(manifest_path)

## 6 · Kontrola

Notebook ověří devět celých pohádek, shodu dimenzí a úplnost všech 36 unikátních párových vzdáleností. Pokud kontrola projde, stáhni tři JSON soubory a načti je do MOUSE CORE.

In [ ]:
assert len(vector_rows) == 9
assert all(len(x["vector"]) == vectors.shape[1] for x in vector_rows)
assert len(geometry) == 9 * 8 // 2
assert [x["card_id"] for x in vector_rows] == list(CARDS.keys())
assert all(x["kind"] == "full_story" for x in vector_rows)
assert np.allclose(cos_matrix, cos_matrix.T)
assert np.allclose(euclidean_matrix, euclidean_matrix.T)
assert all("Karta:" not in d["original_text"] and not re.search(r"(?m)^Text:", d["original_text"]) for d in documents)
print("Kontrola prošla.")
print("9 celých pohádek · 9 skutečných vektorů · 36 unikátních párových vztahů.")
print("Technické řádky Karta:/Text: odstraněny; literární text zachován.")
print("Výstupy: mouse_core_bge_m3_vectors_v2.json, mouse_core_bge_m3_analysis_v2.json, mouse_core_bge_m3_manifest_v2.json")

## 7 · Samostatná referenční lidová pohádka

Lidový text je načten odděleně od devíti autorských karet. Počítá se jeho strukturální profil, jeden celý textový embedding a devět vztahů vůči autorským kartám. Referenční text se nepřimíchává do devítibodové matice.

In [ ]:
REFERENCE = {"id": "kohoutek_slepicka", "name": "O kohoutkovi a slepičce", "path": "doma/primarni_data/reference/lidova_pohadka_kohoutek_a_slepicka.txt"}
reference_raw = urlopen(f"{REPO}/{REFERENCE['path']}").read().decode("utf-8-sig")
reference_text = reference_raw.strip()
assert reference_text.startswith("O kohoutkovi a slepičce")
reference_doc = {"card_id": REFERENCE["id"], "card_name": REFERENCE["name"], "source_file": REFERENCE["path"], "original_text": reference_text}
reference_structure = analyze_structure(reference_doc)
reference_encoded = model.encode([reference_text], batch_size=1, max_length=8192, return_dense=True, return_sparse=False, return_colbert_vecs=False)
reference_vector = np.asarray(reference_encoded["dense_vecs"][0], dtype=np.float32)
reference_pairs = []
for doc, vec in zip(documents, vectors):
    c = cosine(reference_vector, vec)
    reference_pairs.append({"reference_id": REFERENCE["id"], "reference_name": REFERENCE["name"], "card_id": doc["card_id"], "card_name": doc["card_name"], "cosine_similarity": c, "cosine_distance": float(1-c), "angular_distance_degrees": float(np.degrees(np.arccos(np.clip(c,-1,1)))), "euclidean_distance": float(np.linalg.norm(reference_vector-vec))})
display(pd.DataFrame([{"reference": REFERENCE["name"], "characters": reference_structure["characters_including_spaces"], "word_tokens": reference_structure["word_tokens"], "unique_forms": reference_structure["unique_word_forms_casefolded"], "type_token_ratio": reference_structure["type_token_ratio"], "repeated_types": reference_structure["repeated_token_types"], "paragraphs": reference_structure["paragraph_count"], "sentence_end_marks": reference_structure["sentence_ending_mark_count"]}]).set_index("reference"))
display(pd.DataFrame(reference_pairs).sort_values("cosine_similarity", ascending=False))

reference_payload = {"reference": REFERENCE, "model": MODEL_NAME, "vector_dimension": int(len(reference_vector)), "structure": reference_structure, "vector": [float(v) for v in reference_vector], "comparison_to_authored_cards": reference_pairs, "method_note": "Reference folk tale is kept separate from the nine-card authored cohort. Similarity is a model-space relation, not proof of understanding."}
reference_path = OUT / "mouse_core_bge_m3_reference_comparison_v2.json"
reference_path.write_text(json.dumps(reference_payload, ensure_ascii=False, indent=2), encoding="utf-8")

# Doplníme výsledek reference do analýzy a manifestu, aniž bychom měnili devět autorských vektorů.
analysis_path = OUT / "mouse_core_bge_m3_analysis_v2.json"
manifest_path = OUT / "mouse_core_bge_m3_manifest_v2.json"
analysis_existing = json.loads(analysis_path.read_text(encoding="utf-8"))
analysis_existing["reference_folk_tale"] = {"id": REFERENCE["id"], "name": REFERENCE["name"], "source_file": REFERENCE["path"], "structure": reference_structure, "comparison_to_authored_cards": reference_pairs, "separate_vector_file": reference_path.name}
analysis_existing["manifest"]["reference_folk_tale"] = {"name": REFERENCE["name"], "role": "separate_reference_not_part_of_authored_cohort", "source_file": REFERENCE["path"]}
analysis_path.write_text(json.dumps(analysis_existing, ensure_ascii=False, indent=2), encoding="utf-8")
manifest_existing = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest_existing["reference_folk_tale"] = {"name": REFERENCE["name"], "role": "separate_reference_not_part_of_authored_cohort", "source_file": REFERENCE["path"], "comparison_count": len(reference_pairs)}
manifest_path.write_text(json.dumps(manifest_existing, ensure_ascii=False, indent=2), encoding="utf-8")
print("Reference uložena odděleně. Devět autorských vektorů zůstává beze změny.")
print("Nový soubor:", reference_path)

## 8 · Stažení výsledků

ZIP obsahuje původní tři soubory devítikaretního pilotu doplněné o samostatnou referenční pohádku a její vektor.

In [ ]:
import zipfile
zip_path = OUT / "MOUSE_CORE_v2_vysledky.zip"
files_to_zip = [
    OUT / "mouse_core_bge_m3_vectors_v2.json",
    OUT / "mouse_core_bge_m3_analysis_v2.json",
    OUT / "mouse_core_bge_m3_manifest_v2.json",
    OUT / "mouse_core_bge_m3_reference_comparison_v2.json"
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path in files_to_zip:
        if path.exists():
            zipf.write(path, arcname=path.name)
        else:
            print("CHYBÍ:", path.name)
print("ZIP připraven:", zip_path)
from google.colab import files
files.download(str(zip_path))